# 08b — Análise secundária: LSTM (global, univariada e multivariada)

**Lê:** treino, teste (trava, *secundaria*), série completa, `exogenas_semanais`.  
**Grava:** `secundaria_pred_lstm`.

Separado do 08a porque usa PyTorch (no Colab, pode ativar GPU sem afetar os outros notebooks). Épocas escolhidas num holdout interno do **treino** (últimas `LSTM_HOLDOUT_WEEKS` semanas-alvo).

In [ ]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "08b_secundaria_lstm"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

In [ ]:
from tcc_fluxo import secundaria as S

spec_json = A.carregar_json("painel_spec")
TEST_END = pd.Timestamp(A.carregar_json("corte")["TEST_END"])
exog = A.carregar_df("exogenas_semanais")
train = A.carregar_treino()
test = A.carregar_teste(finalidade="secundaria")
serie = A.carregar_serie_completa(finalidade="secundaria")
MACRO = spec_json["MACRO_FEATURES"]
sec_exog, _, _ = S.preparar_exogenas(exog, MACRO, C.SEC_EXOG_CAL, C.PANEL_START, C.TRAIN_TARGET_END, TEST_END)

janelas = S.JanelasLSTM(serie, C.LSTM_SEQ_LEN, sec_exog, MACRO)
static_multi = MACRO + spec_json["HOLIDAY_FEATURES"] + spec_json["COMMERCIAL_FEATURES"] + ["target_week_sin", "target_week_cos"]
VARIANTS = {"LSTM_global": None,                                   # v5: receita + pedidos + seno/cosseno
            "LSTM_uni": {"seq": ["revenue"], "static": []},
            "LSTM_multi": {"seq": ["revenue", "orders"], "static": static_multi}}

In [ ]:
preds, info, y_scale = S.rodar_lstms(janelas, train, test, VARIANTS, C)
out = test[["seller_id", "week_start", "target_week"]].copy()
for m, p in preds.items():
    out[f"pred_{m}"] = p
A.salvar_df(out, "secundaria_pred_lstm")
A.tabela(info, "secundaria_lstm_treino.csv"); display(info)

In [ ]:
A.finalizar()